<b><font size="6" color="#E8800A">Week 5 · Model Selection</font></b><br>

A validation score guides model choices, so it cannot also provide an independent
assessment of the selected model. This notebook separates those jobs using a
train, validation and test split, then repeats the training and validation
comparison through cross-validation. Every comparison inherits the preprocessing
recipe recorded in Weeks 3 and 4.

<div class="alert alert-block alert-info">

## Table of Contents<a class="anchor" id="toc"></a>
### [<font color='#E8800A'>1 - Data Setup</font>](#setup)
* [<font color='#E8800A'>1.1. - Loading Libraries</font>](#libraries)
* [<font color='#E8800A'>1.2. - Loading the Dataset and Recipe</font>](#data-loading)
### [<font color='#E8800A'>2 - Holdout: Training, Validation and Test</font>](#train-test)
* [<font color='#E8800A'>2.1. - Reserving the Test Set</font>](#train-test-split)
* [<font color='#E8800A'>2.2. - Creating Training and Validation Sets</font>](#three-way-split)
### [<font color='#E8800A'>3 - Cross-Validation Techniques</font>](#cross-validation)
* [<font color='#E8800A'>3.1. - K-Fold Cross-Validation</font>](#kfold)
* [<font color='#E8800A'>3.2. - Repeated K-Fold Cross-Validation</font>](#repeated-kfold)
* [<font color='#E8800A'>3.3. - Leave-One-Out Cross-Validation</font>](#loo)
* [<font color='#E8800A'>3.4. - Group-Aware and Time-Aware Splitting</font>](#stratified)
* [<font color='#E8800A'>3.5. - Which Design Estimated Best?</font>](#frame-comparison)
### [<font color='#E8800A'>4 - Comparing Models</font>](#model-comparison)
* [<font color='#E8800A'>4.1. - Decision Tree Classifier</font>](#decision-tree)
* [<font color='#E8800A'>4.2. - Model Comparison Results</font>](#comparison-results)
### [<font color='#E8800A'>5 - Hyperparameter Tuning</font>](#hyperparameter-tuning)
* [<font color='#E8800A'>5.1. - What are Hyperparameters?</font>](#what-are-hyperparameters)
* [<font color='#E8800A'>5.2. - Hyperparameter Search with Holdout Validation</font>](#grid-search)
* [<font color='#E8800A'>5.3. - Grid Search with Cross-Validation</font>](#grid-search-cv)
* [<font color='#E8800A'>5.4. - Using Pipeline with GridSearchCV</font>](#pipeline)
* [<font color='#E8800A'>5.5. - The Two Execution Schemas</font>](#execution-schemas)
### [<font color='#E8800A'>Gold Standard: Nested Cross-Validation</font>](#nested-cv)
### [<font color='#E8800A'>The Complete Model Selection Workflow</font>](#workflow)
* [<font color='#E8800A'>Key Takeaways</font>](#takeaways)
### Optional (Advanced)
* [<font color='#E8800A'>Scikit-learn Hyperparameter Optimization Tools</font>](#optional-tools)
</div>

<a class="anchor" id="setup">

## <font color='#E8800A'>1. Data Setup</font>
</a>

The cleaned data and recorded recipe provide the starting point for model
selection. Loading them restores the earlier decisions before any split is made.

<a class="anchor" id="libraries">

### <font color='#E8800A'>1.1. Loading Libraries</font>
</a>

__`Step 1`__ Import the libraries used for splitting, fitting and comparison.

[Back to TOC](#toc)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

from time import perf_counter

from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import (
    GridSearchCV,
    GroupKFold,
    KFold,
    LeaveOneOut,
    RepeatedKFold,
    RepeatedStratifiedKFold,
    StratifiedKFold,
    TimeSeriesSplit,
    cross_validate,
    train_test_split,
)
from sklearn.pipeline import Pipeline

from course_helpers import PLOT_BLUE, PLOT_ORANGE

import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 33
np.random.seed(RANDOM_STATE)  # For reproducibility

import sys
from pathlib import Path
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

from preprocessing import (
    CachedSearchCV,
    PreparedEstimator,
    classification_preprocessor,
    load_classification,
)
CLASSIFICATION_RECIPE_LOG = "../../logs/week_04_feature_work_classification_log.json"


<a class="anchor" id="data-loading">

### <font color='#E8800A'>1.2. Loading the Dataset and Recipe</font>
</a>

`champions.csv` contains the rows retained in Week 3, with missing values still
open. The Week 4 log includes the earlier dtype, column-role, filling and
transformation decisions, followed by its encoding, scaling and selection rules.
`load_classification` reads that log and restores the recorded dtypes while
loading the CSV. The functions in `preprocessing.py` rebuild its supported rules
for fitting inside each new split.

__`Step 2`__ Load `champions.csv` with its Week 4 recipe and recorded dtypes.

Run this notebook from its own folder inside the course
repository: that is what makes the `data/` paths below work. If you
downloaded this file on its own from Moodle, move it into the repository before
you run it.

In [ ]:
champions, classification_recipe = load_classification(
    '../../data/interim/champions.csv',
    CLASSIFICATION_RECIPE_LOG,
)
# The loader restores the nullable dtypes recorded in the log.
print(f'{champions.shape[0]:,} rows x {champions.shape[1]} columns')
champions.head()

__`Step 3`__ Separate the target `Outcome` from the inherited feature columns.

The recipe names the categorical and numeric features selected in the earlier
weeks. `RecordID` and `Athlete Id` identify rows and are absent from those lists.
Copy both the feature frame and the target so later edits do not alter `champions`.

In [ ]:
# Outcome is the classification target. Keep independent copies.
target = champions['Outcome'].copy()
categorical_cols = list(classification_recipe['categorical'])
numerical_cols = list(classification_recipe['numeric'])
feature_columns = [column for column in champions.columns
                   if column in categorical_cols + numerical_cols]
data = champions.loc[:, feature_columns].copy()

print('Target:', target.name)
print(f'{len(categorical_cols)} categorical + {len(numerical_cols)} numeric features')

In [ ]:
# Read the inherited settings; this week changes the evaluation design.
print('Fill rules:', classification_recipe['fill'])
print('log1p columns:', classification_recipe['log1p'])
print('Encoding:', classification_recipe['encoding'])
print('Scaling:', classification_recipe['scaler'])
print('Selection:', classification_recipe['strategy']['kind'])

<a class="anchor" id="train-test">

## <font color='#E8800A'>2. Holdout: Training, Validation and Test</font>
</a>

The training set supplies fitted values and model parameters. Validation scores
guide model and hyperparameter choices; the test set assesses the resulting
choice after selection is complete. Splitting the raw feature rows first keeps
both held-out sets outside every preprocessing fit.

<a class="anchor" id="train-test-split">

### <font color='#E8800A'>2.1. Reserving the Test Set</font>
</a>

Reserve 20% of the rows for final testing and retain 80% for training and
validation. Stratifying by `Outcome` preserves the class proportions in both
parts. A fixed random seed makes this particular split reproducible.

[Back to TOC](#toc)


<div class="alert alert-block alert-info">
<a href='https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html'>sklearn.model_selection.train_test_split(*arrays, test_size=None, train_size=None, random_state=None, shuffle=True, stratify=None)</a>

**Definition:**  
Split arrays or matrices into random train and test subsets.

**Common Parameters:**  
- `test_size`: Proportion of the dataset to include in the test split (e.g., 0.2 for 20%)
- `train_size`: Proportion of the dataset to include in the train split
- `random_state`: Controls the shuffling for reproducible output
- `shuffle`: Whether to shuffle the data before splitting
- `stratify`: If not None, data is split in a stratified fashion using this as class labels

**Returns:**  
- Splitting of inputs into train-test sets
</div>

__`Step 4`__ Reserve 20% for testing; keep the other 80% in `X_train_val` and `y_train_val`.

In [ ]:
X_train_val, X_test, y_train_val, y_test = train_test_split(data, 
                                                    target, 
                                                    test_size=0.2, 
                                                    random_state=RANDOM_STATE, 
                                                    shuffle=True, 
                                                    stratify=target
                                                   )

The test labels remain outside model selection. A project competition may supply this test partition separately; here we reserve it from the labelled data.

<a class="anchor" id="three-way-split">

### <font color='#E8800A'>2.2. Creating Training and Validation Sets</font>
</a>

Split the remaining 80% once more. Taking one quarter of it for validation
leaves 60% of all rows for training, 20% for validation and the reserved 20%
for testing. Validation may guide choices; it may not supply fitted fill values,
categories, scaling statistics or selected features.

__`Step 5`__ Split `X_train_val` and `y_train_val`, stratifying by their own target labels.

In [ ]:
# DO IT
X_train, X_val, y_train, y_val = train_test_split(X_train_val,
                                                  y_train_val,
                                                  test_size = 0.25,
                                                  random_state = RANDOM_STATE,
                                                  shuffle=True,
                                                  stratify=y_train_val
)

__`Step 6`__ Check the proportion of data for each dataset. _(written for you)_

In [ ]:
print(f'train: {len(y_train) / len(target):.0%} | '
      f'validation: {len(y_val) / len(target):.0%} | '
      f'test: {len(y_test) / len(target):.0%}')

__`Step 7`__ Fit the inherited recipe on training rows and transform all three sets.

`classification_preprocessor` packages the filling, encoding, transformation,
scaling and selection logic used in Weeks 3 and 4. Its argument is the logged
recipe, so the settings arrive with the data. `fit_transform(X_train, y_train)`
learns the fitted values and selected columns from training rows alone;
`transform` applies that same fit to validation and test rows.

In [ ]:
holdout_preprocessor = classification_preprocessor(classification_recipe)
X_train_prepared = holdout_preprocessor.fit_transform(X_train, y_train)
X_val_prepared = holdout_preprocessor.transform(X_val)
X_test_prepared = holdout_preprocessor.transform(X_test)

print('encoded columns:', len(holdout_preprocessor.encoded_names_))
print('kept by the selection:', X_train_prepared.shape[1])

__`Step 8`__ Fit a classifier on the training rows, then score it on the validation rows and on the test rows.

The classifier sees only `X_train_prepared` and `y_train`. Two scores come out of
that one fit, and they do different jobs. The validation score is the estimate:
every choice made below is allowed to read it. The test score is the quantity
that estimate is trying to predict, and no choice below is allowed to read it.

Every score in this notebook is the F1 of the positive class, a win
(`Outcome` = 1). Below the first result the cell prints the share of wins in
the test rows and the F1 of predicting a win for every one of them, the floor a
classifier has to clear.

In [ ]:
holdout_model = LogisticRegression(max_iter=1000)
holdout_model.fit(X_train_prepared, y_train)
holdout_val_predictions = holdout_model.predict(X_val_prepared)
holdout_test_predictions = holdout_model.predict(X_test_prepared)

holdout_score = f1_score(y_val, holdout_val_predictions)
holdout_test_score = f1_score(y_test, holdout_test_predictions)

# Section 3 changes the validation design and keeps the model, so every design
# from here on records what it estimated beside what it was estimating, and
# reprints the whole record.
frame_log = [{'frame': 'holdout', 'validation F1 is': 'untuned',
              'rows': len(y_train), 'folds': 1,
              'train F1': float(f1_score(y_train, holdout_model.predict(X_train_prepared))),
              'validation F1': float(holdout_score),
              'test F1': float(holdout_test_score),
              'model': repr(holdout_model)}]


def show_log(log):
    """Print every row of a log, and how far each estimate landed from its test F1."""
    table = pd.DataFrame(log)
    table['estimate minus truth'] = table['validation F1'] - table['test F1']
    table['absolute error'] = table['estimate minus truth'].abs()
    # The model's name is the longest column, so it goes last, left-aligned.
    table = table[[column for column in table if column != 'model'] + ['model']]
    width = table['model'].str.len().max()
    print(table.to_string(
        index=False,
        formatters={'train F1': '{:.4f}'.format,
                    'validation F1': '{:.4f}'.format,
                    'test F1': '{:.4f}'.format,
                    'estimate minus truth': '{:+.4f}'.format,
                    'absolute error': '{:.4f}'.format,
                    'model': lambda name: name.ljust(width)}))


print(f'one holdout split, {len(y_train)} train / {len(y_val)} validation'
      f' / {len(y_test)} test rows\n')
show_log(frame_log)

always_win = np.ones(len(y_test), dtype=int)
print(f'\nshare of wins in the test rows: {y_test.mean():.0%}')
print(f'F1 of predicting a win for every test row: {f1_score(y_test, always_win):.4f}')

<a class="anchor" id="cross-validation">

## <font color='#E8800A'>3. Cross-Validation Techniques</font>
</a>

One holdout score depends on the particular training and validation rows drawn.
Cross-validation repeats that comparison over several splits of `X_train_val`
and `y_train_val`. Each fold refits the inherited recipe and a logistic regression
using only its training rows, then scores its validation rows. The reserved test
set remains outside these comparisons.

<a class="anchor" id="kfold">

### <font color='#E8800A'>3.1. K-Fold Cross-Validation</font>
</a>

K-Fold divides the available rows into K folds. Each fold supplies validation
rows once, while the other K-1 folds supply training rows. Averaging those scores
reduces dependence on one split, although the folds share training observations
and therefore do not produce independent estimates.

<img src="https://scikit-learn.org/stable/_images/grid_search_cross_validation.png" alt="Training and validation folds" style="width: 500px;"/>

[Back to TOC](#toc)


Keep the classifier and the inherited recipe fixed while comparing splitters, so the validation design is what changes.

The `KFold` class provides a way to split your dataset into K consecutive folds for cross-validation. Each fold is used once as a validation while the K-1 remaining folds form the training set.

<div class="alert alert-block alert-info">
<a href='https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.KFold.html'>sklearn.model_selection.KFold(n_splits=5, shuffle=False, random_state=None)</a>

**Definition:**  
K-Folds cross-validator. Provides train/test indices to split data into train/test sets.

**Common Parameters:**  
- `n_splits`: Number of folds (must be at least 2)
- `shuffle`: Whether to shuffle the data before splitting into batches
- `random_state`: Controls the randomness of shuffle (for reproducibility)

**Common Methods:**  
- `split(X, y=None, groups=None)`: Generate indices to split data into training and test set
- `get_n_splits(X=None, y=None, groups=None)`: Returns the number of splitting iterations

**Usage:**  
Splits the dataset into K consecutive folds. Each fold is used once as validation while the K-1 remaining folds form the training set.
</div>

Alternatively, we can use **Stratified K-Fold Cross-Validation**, which ensures that each fold maintains the same class distribution as the entire dataset. This is particularly important for imbalanced classification problems.

<div class="alert alert-block alert-info">
<a href='https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.StratifiedKFold.html'>sklearn.model_selection.StratifiedKFold(n_splits=5, shuffle=False, random_state=None)</a>

**Definition:**  
Stratified K-Folds cross-validator. Provides train/test indices to split data into train/test sets while preserving the percentage of samples for each class.

**Common Parameters:**  
- `n_splits`: Number of folds (must be at least 2)
- `shuffle`: Whether to shuffle each class's samples before splitting into batches
- `random_state`: Controls the randomness of shuffle (for reproducibility)

**Common Methods:**  
- `split(X, y, groups=None)`: Generate indices to split data into training and test set
- `get_n_splits(X=None, y=None, groups=None)`: Returns the number of splitting iterations

**Usage:**  
Variation of K-Fold that returns stratified folds - each fold preserves the same class distribution as the complete dataset. Particularly important for imbalanced classification problems.
</div>

__`Step 9`__ Do steps 7 and 8 again as one object, and check the two agree.

In [ ]:
# Steps 7 and 8 fitted the recipe, transformed three matrices, then
# fitted a classifier on the first of them. PreparedEstimator is a custom
# class in preprocessing.py that does exactly that.
prepared = PreparedEstimator(
    classification_preprocessor(classification_recipe),
    LogisticRegression(max_iter=1000),
).fit(X_train, y_train)

print(f'by hand, steps 7-8 : {holdout_score:.4f}')
print(f'PreparedEstimator  : {f1_score(y_val, prepared.predict(X_val)):.4f}')

__`Step 10`__ Define two helpers. `avg_score(method, X, y, model=None, groups=None, pooled=False)` scores F1 on both sides of every fold and returns the winning model refitted on every row. `record(log, result, X_test, y_test)` scores that model once on the test rows and adds its row to a log.

In [ ]:
def avg_score(method, X, y, model=None, groups=None, pooled=False):
    """Cross-validate one model, or choose among several, and return it fitted.

    `model` is an unfitted estimator, an untuned logistic regression by
    default, or a list of candidates. Every fold refits the recipe and the
    model on that fold's training rows alone and scores both sides by F1.
    With a list, the first candidate with the highest mean validation F1 wins.

    `groups` reaches the splitters that need it, such as GroupKFold. With
    `pooled`, validation F1 is computed once over every held-out prediction,
    which folds of a single row need.

    Returns a dict: the winner's 'name', its per-fold 'train' and
    'validation' F1, the 'frame', 'rows' and 'folds' it was measured on, and
    the winner refitted on every row of X as 'model', ready to predict.
    """
    if isinstance(model, list):
        candidates = model
    else:
        candidates = [LogisticRegression(max_iter=1000) if model is None else model]
    # One set of folds for every candidate, so they compete on the same rows.
    folds = list(method.split(X, y, groups))

    best = None
    for candidate in candidates:
        score_train, held_out = [], []
        for train_index, val_index in folds:
            X_train, X_val = X.iloc[train_index], X.iloc[val_index]
            y_train, y_val = y.iloc[train_index], y.iloc[val_index]

            fitted = PreparedEstimator(
                classification_preprocessor(classification_recipe),
                clone(candidate),
            ).fit(X_train, y_train)
            score_train.append(f1_score(y_train, fitted.predict(X_train)))
            held_out.append((y_val, fitted.predict(X_val)))

        if pooled:
            held_out = [(pd.concat([truth for truth, _ in held_out]),
                         np.concatenate([guess for _, guess in held_out]))]
        score_val = [f1_score(truth, guess) for truth, guess in held_out]
        # The other metrics are reported beside F1; only F1 chooses.
        other = {label: np.mean([metric(truth, guess) for truth, guess in held_out])
                 for label, metric in (('accuracy', accuracy_score),
                                       ('precision', precision_score),
                                       ('recall', recall_score))}

        name = repr(candidate)
        spread = ' (pooled)' if pooled else f' +/- {np.std(score_val):.4f}'
        print(f'train F1 {np.mean(score_train):.4f} | validation accuracy '
              f"{other['accuracy']:.4f}, precision {other['precision']:.4f}, "
              f"recall {other['recall']:.4f}, F1 {np.mean(score_val):.4f}{spread} | {name}")
        # Strictly greater, so on a tie the earlier candidate keeps the lead.
        if best is None or np.mean(score_val) > np.mean(best['validation']):
            best = {'name': name, 'candidate': candidate,
                    'train': score_train, 'validation': score_val}

    best['model'] = PreparedEstimator(
        classification_preprocessor(classification_recipe),
        clone(best.pop('candidate')),
    ).fit(X, y)
    best.update(frame=type(method).__name__, rows=len(y), folds=len(folds))
    return best


def record(log, result, X_test, y_test, validation_is='untuned'):
    """Score a returned model once on the test rows, add its row and print the log."""
    row = {
        'frame': result['frame'],
        'validation F1 is': validation_is,
        'rows': result['rows'],
        'folds': result['folds'],
        'train F1': float(np.mean(result['train'])),
        'validation F1': float(np.mean(result['validation'])),
        'test F1': float(f1_score(y_test, result['model'].predict(X_test))),
        'model': result['name'],
    }
    # Running a cell again replaces its row rather than adding a second one.
    key = (row['frame'], row['validation F1 is'], row['model'])
    log[:] = [old for old in log
              if (old['frame'], old['validation F1 is'], old['model']) != key]
    log.append(row)
    show_log(log)

`PreparedEstimator` comes from `preprocessing.py`, which is where every step Week 4 settled already lives. Each fold builds a fresh one from the same logged recipe, so only that fold's training rows fit the fills, categories, scale, selected columns and classifier; scoring applies all of them to the validation rows without refitting anything.

__`Step 11`__ Create a KFold Instance where the number of splits is 10 (*n_splits*) and name it as __kf__

In [ ]:
kf = ...  # <-- CODE HERE


__`Step 12`__ Create a StratifiedKFold Instance where the number of splits is 10 (*n_splits*) and name it as __skf__

In [ ]:
skf = ...  # <-- CODE HERE


__`Step 13`__ Call the function __avg_score__ and check the average score for the training and validation sets using __kf__, `record` the result in `frame_log`, and display the per-fold validation F1 in `kf_result['validation']`

In [ ]:
kf_result = ...  # <-- CODE HERE
record(...)  # <-- CODE HERE
...  # <-- CODE HERE


__`Step 14`__ Call the function __avg_score__ and check the average score for the train and the validation sets using __skf__, then `record` the result in `frame_log`

In [ ]:
skf_result = ...  # <-- CODE HERE
record(...)  # <-- CODE HERE


**The single split was reading its own luck.** The holdout in Section 2
scored **0.8379**; ten folds of the same 3,200 rows average **0.8527**, and
stratifying them gives **0.8546**. One and a half points of F1 separate the two
answers, and the model settings did not change between them: only which rows,
and how many, trained and validated it.

The fold scores printed above are a reason to prefer the average. They run
from **0.8269** to **0.8750**, a spread of almost five points, so a single draw
from that range is a poor estimate of where the middle is. **Averaging ten of
them does not make the model better**; it makes the number you quote more
reliable than a single estimate.

Stratification is the smaller of the two effects here, because `Outcome` is
not far from balanced. On a rarer class it is the larger one, since an
unstratified fold can end up with too few positives to score at all.

The table printed under each of those scores already shows the holdout
estimate landing **0.0102** below the test score while ten folds land
**0.0053** above it, so the cheapest design is not merely noisier here, it
missed by more. On both ten-fold designs the `train F1` column sits within
half a point of validation, so this model is not memorising its training rows.

<a class="anchor" id="repeated-kfold">

### <font color='#E8800A'>3.2. Repeated K-Fold Cross-Validation</font>
</a>

Repeated K-Fold repeats the entire K-Fold process several times, drawing different random splits each time. Five folds and ten repeats trains and evaluates the model fifty times.

**Why use it? How is this different from a K-fold of 50 folds?** 

Take a dataset of 100 samples. A 50-fold cross-validation makes folds of 2 samples, so each iteration trains on 98 rows and scores 2 of them. A 5-fold cross-validation repeated 10 times makes folds of 20, so each iteration trains on 80 rows and scores 20. Both fit fifty models, and only one of them scores enough rows at a time for a fold score to carry information.
    
**Trade-off**: More computationally expensive than regular K-Fold, potentially an overkill in larger datasets.

<div class="alert alert-block alert-info">
<a href='https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.RepeatedKFold.html'>sklearn.model_selection.RepeatedKFold(n_splits=5, n_repeats=10, random_state=None)</a>

**Definition:**  
Repeated K-Fold cross validator. Repeats K-Fold n times with different randomization in each repetition.

**Common Parameters:**  
- `n_splits`: Number of folds (must be at least 2)
- `n_repeats`: Number of times cross-validator needs to be repeated
- `random_state`: Controls the randomness of shuffle (for reproducibility)

**Common Methods:**  
- `split(X, y=None, groups=None)`: Generates indices to split data into training and test set
- `get_n_splits(X=None, y=None, groups=None)`: Returns the number of splitting iterations (n_splits * n_repeats)

**Usage:**  
Provides more robust estimates by repeating K-Fold multiple times with different splits, reducing variance in performance estimates.
</div>

__`Step 15`__ Create a RepeatedKFold Instance where the number of splits is 6 (`n_splits=6`) and the number of times cross-validator needs to be repeated is 2 (`n_repeats=2`), with `random_state=RANDOM_STATE` so the folds are the same on every run, and name it as __rkf__

In [ ]:
rkf = ...  # <-- CODE HERE


__`Step 16`__ Call the function __avg_score__ and check the average score for the training and validation sets using __rkf__, `record` the result in `frame_log`, and display the per-fold validation F1 in `rkf_result['validation']`

In [ ]:
rkf_result = ...  # <-- CODE HERE
record(...)  # <-- CODE HERE
...  # <-- CODE HERE


<a class="anchor" id="loo">

### <font color='#E8800A'>3.3. Leave-One-Out Cross-Validation (LOOCV)</font>
</a>

Leave-One-Out is the extreme case of K-Fold where K equals the number of
samples. Each iteration trains on every row except one and scores that one
held-out row.

That design belongs to very small datasets, and this one is not small. The
cell below runs it anyway, over all 3,200 training and validation rows, so the
comparison stays controlled: every design in this section except
`TimeSeriesSplit`, which has to drop the undated rows, is then handed the same
3,200 rows and ships the same model, and only the estimate differs.

**Expect this cell to take around thirty minutes.** It performs 3,200 fold
fits, each fitting its encoder, imputer, scaler and model on 3,199 rows before
scoring the single row it held back. Section 3.1 bought its estimate with ten
fold fits.

One row is too few to score. F1 is a property of a set of predictions. A
held-out row that was not a win, predicted as not a win, has no F1 of its own,
and scoring such rows 0 would turn the average into the share of all 3,200 rows
that are wins the model caught. So `avg_score` is told to pool: it scores one F1
over all 3,200 held-out predictions.

__`Step 17`__ Do the same with `LeaveOneOut`, passing `pooled=True` to __avg_score__, and record the result in `frame_log`.

<div class="alert alert-block alert-info">
<a href='https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.LeaveOneOut.html'>sklearn.model_selection.LeaveOneOut()</a>

**Definition:**  
Leave-One-Out (LOO) cross-validator. Provides train/test indices to split data where each sample is used once as test set while remaining samples form the training set.

**Common Methods:**  
- `split(X, y=None, groups=None)`: Generate indices to split data into training and test set
- `get_n_splits(X)`: Returns the number of splitting iterations (equals number of samples)

**Usage:**  
Special case of K-Fold where k=n (number of samples). Each iteration uses exactly one sample for testing. Very thorough but computationally expensive for large datasets.
</div>

In [ ]:
# LOO is appropriate only for small data. Its folds hold one row each, so
# avg_score pools the held-out predictions into a single F1.
loo = LeaveOneOut()
print('LOO fold fits:', loo.get_n_splits(X_train_val),
      '-- this takes about thirty minutes on my computer.')
loo_result = avg_score(loo, X_train_val, y_train_val, pooled=True)
record(frame_log, loo_result, X_test, y_test)

<a class="anchor" id="stratified">

### <font color='#E8800A'>3.4. Group-Aware and Time-Aware Splitting</font>
</a>

Scikit-learn features multiple options to select your model and, syntax-wise, they have a similar way of being used. You can check the [full list](https://scikit-learn.org/stable/api/sklearn.model_selection.html) below.

| Class Name | Description | Documentation Link |
|-----------|-------------|-------------------|
| **train_test_split** | Split arrays or matrices into random train and test subsets | [Documentation](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html) |
| **KFold** | K-Fold cross-validator | [Documentation](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.KFold.html) |
| **StratifiedKFold** | Class-wise stratified K-Fold cross-validator | [Documentation](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.StratifiedKFold.html) |
| **RepeatedKFold** | Repeated K-Fold cross validator | [Documentation](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.RepeatedKFold.html) |
| **RepeatedStratifiedKFold** | Repeated class-wise stratified K-Fold cross validator | [Documentation](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.RepeatedStratifiedKFold.html) |
| **LeaveOneOut** | Leave-One-Out cross-validator | [Documentation](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.LeaveOneOut.html) |
| **LeavePOut** | Leave-P-Out cross-validator | [Documentation](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.LeavePOut.html) |
| **GroupKFold** | K-fold iterator variant with non-overlapping groups | [Documentation](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GroupKFold.html) |
| **StratifiedGroupKFold** | Class-wise stratified K-Fold iterator variant with non-overlapping groups | [Documentation](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.StratifiedGroupKFold.html) |
| **LeaveOneGroupOut** | Leave One Group Out cross-validator | [Documentation](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.LeaveOneGroupOut.html) |
| **LeavePGroupsOut** | Leave P Group(s) Out cross-validator | [Documentation](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.LeavePGroupsOut.html) |
| **ShuffleSplit** | Random permutation cross-validator | [Documentation](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.ShuffleSplit.html) |
| **StratifiedShuffleSplit** | Class-wise stratified ShuffleSplit cross-validator | [Documentation](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.StratifiedShuffleSplit.html) |
| **GroupShuffleSplit** | Shuffle-Group(s)-Out cross-validation iterator | [Documentation](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GroupShuffleSplit.html) |
| **TimeSeriesSplit** | Time Series cross-validator | [Documentation](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.TimeSeriesSplit.html) |
| **PredefinedSplit** | Predefined split cross-validator | [Documentation](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.PredefinedSplit.html) |
| **check_cv** | Input checker utility for building a cross-validator | [Documentation](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.check_cv.html) |


### Try it: our data has both a group and a time structure

`Competition` places every athlete in a competition level, such as `Local Match`
or `Federation League`, and `Edition` orders them in time: the two structures
that `GroupKFold` and `TimeSeriesSplit` in the table above are built for. Which
splitter is correct here is settled by the **structure of the data** and the
question asked, not by which one scores better: athletes from one level share
whatever that level does to the outcome, and a later edition cannot be predicted
from its own future.

`Athlete Id` would be the obvious group key, except the spine was deduplicated on
it, so every athlete appears exactly once and grouping by it would be identical to
a plain `KFold`. The group key has to be a structure the data still contains.

The next two cells take one structure each, on the same `X_train_val` and
`y_train_val` rows, and each prints the evidence that its splitter kept its
promise.

**The group structure.** `GroupKFold` guarantees that no competition
level supplies rows to both sides of a fold. The cell prints the levels each fold
validates on and checks the guarantee on every fold before anything is fitted:
each count of shared levels should be zero. It also prints, per fold, the F1 of
predicting a win for every row.

In [ ]:
# The key is already in the frame: no re-reading the file.
groups_train_val = (
    champions.loc[X_train_val.index, "Competition"]
    .astype('string')
    .fillna('<missing competition>')
)
gkf = GroupKFold(n_splits=5)

# Held-out levels, shared levels and the always-win F1, per fold.
held_out, shared_levels, win_floor = [], [], []
for train_idx, val_idx in gkf.split(X_train_val, y_train_val, groups=groups_train_val):
    train_levels = set(groups_train_val.iloc[train_idx])
    val_levels = set(groups_train_val.iloc[val_idx])
    held_out.append(sorted(val_levels))
    shared_levels.append(len(train_levels & val_levels))
    y_fold = y_train_val.iloc[val_idx]
    win_floor.append(round(f1_score(y_fold, np.ones(len(y_fold), dtype=int)), 4))

print("Competition groups:", groups_train_val.nunique())
for fold, levels in enumerate(held_out, start=1):
    print(f"fold {fold} validates on: {', '.join(levels)}")
print("Levels shared between training and validation, per fold:", shared_levels)
print("F1 of predicting a win for every row, per fold:", win_floor, "\n")

group_result = avg_score(gkf, X_train_val, y_train_val, groups=groups_train_val)
record(frame_log, group_result, X_test, y_test)
group_result['validation']

The fold scores run from **0.4268** to **0.8684**, a spread no random design in
this section came near, and they average **0.6684** against the **0.8527** that
ten random folds reported on the very same rows. That average is also below the
floor: predicting a win for every athlete scores between **0.6873** and
**0.8405** on these folds, and the model beats that constant on two of the
five.

Each fold validates on whole competition levels the model never trained on. A
random fold puts athletes from one level on both sides of the division, so the
model learns what that level does to the outcome through its `Competition`
column. `GroupKFold` removes that, and the drop measures what knowing an
athlete's competition level was worth.

The table disagrees because its test rows were drawn by a random split too, so
every competition level in them also appears in the training rows, and they are
scored under the easier question. The `estimate minus truth` column can only
compare designs that are estimating the same thing, and this one is not.

A missing `Competition` is kept here as its own disclosed group rather than
dropped. Rows with no recorded level are still rows, and putting them together
keeps them from being split across a fold boundary by accident.

**The time structure.** `TimeSeriesSplit` guarantees instead that no
fold's training rows come after its validation rows in the sorted order. That
requires an ordering, so the rows are sorted by `Edition` first, and rows whose
`Edition` is unknown are reported and excluded: a row with no position in the
chronology cannot be placed on either side of a time boundary.

In [ ]:
dates_train_val = champions.loc[X_train_val.index, "Edition"]
known_time = dates_train_val.notna()
# kind='stable' keeps the rows of one edition in their current order.
time_order = dates_train_val[known_time].sort_values(kind='stable').index
X_time = X_train_val.loc[time_order]
y_time = y_train_val.loc[time_order]

print('Rows excluded because Edition is unknown:', int((~known_time).sum()))

tscv = TimeSeriesSplit(n_splits=5)
edition_time = dates_train_val.loc[time_order]
for fold, (train_idx, val_idx) in enumerate(tscv.split(X_time), start=1):
    train_editions = sorted({int(edition) for edition in edition_time.iloc[train_idx]})
    val_editions = sorted({int(edition) for edition in edition_time.iloc[val_idx]})
    print(f"fold {fold}: trains on editions {train_editions}, validates on {val_editions}")
print()

time_result = avg_score(tscv, X_time, y_time)
record(frame_log, time_result, X_test, y_test)
time_result['validation']

Each fold trains on a prefix of the chronology and validates on what comes
next, so no model here is trained on a later edition than the ones it is scored
on. A random `KFold` ignores that order entirely.

The chronological folds average **0.8373**, one and a half points under the ten
random folds and nowhere near the collapse the group split produced. The
explanation is in the design of this particular demonstration rather than in
the method.

The fold editions printed above show it. Four editions cannot fill five folds
one season each, so the split cuts *inside* editions, and every fold trains and
validates on rows from at least one shared season: the last two validate on
2022 rows beside 2022 training rows. The ordering guarantee holds, but the folds
are not one-edition-each, and with four seasons and five folds that is
unavoidable.

A split that held out a whole season would ask a harder question, and this
notebook does not measure it.

<a class="anchor" id="frame-comparison"></a>

### 3.5. Which Design Estimated Best?

The table has been growing under every design in this section, one row at a
time. The validation column is what each design claimed before the answer was
available, and the test column is the answer.

In [ ]:
show_log(frame_log)

**The test column barely moves.** Five designs report exactly **0.8474**.
That is not a bug in the table. A test score comes from refitting the same
logistic regression on every row the design was given, so five designs handed
the same 3,200 rows produce one identical fitted model and therefore one
identical score. Changing how you validate changes what you believe about a
model. It does not change the model.

That repetition is what makes this a controlled comparison, because one fixed
truth leaves every difference in the `absolute error` column to the designs. The
test column moves only where an input moved: `TimeSeriesSplit` reports
**0.8462** on the 3,181 rows left after the undated ones were excluded, and the
holdout row reports **0.8481** from a model fitted on its 2,400 training rows
alone.

Of the designs that ask the test split's question, the single holdout missed by the most, **0.0102**, and the four random multi-fold designs, Leave-One-Out included, by **0.0072** or less. That order is not evidence on its own. The test score is one
measurement on 800 rows, and the ten-fold printouts above spread by **0.0145**
to **0.0170**, one standard deviation, more than any gap in the column.

The case for repeating the split rests on that spread instead. A holdout
estimate rests on the 800 rows it scored once, a single draw from that spread.
A ten-fold estimate rests on 3,200 scored rows, because every row serves as a
validation row exactly once, and an average over more scored rows carries less
of any one split's luck.

**Leave-One-Out is a disappointment.** It fitted 3,200 models, one per row, and took about thirty minutes to report **0.8509**, which misses by **0.0035**. Twelve repeated folds took a few seconds and missed by less. Leave-One-Out is not wrong here, and its fold models train on 3,199 rows, so it is the one design in this section whose folds are essentially the model being deployed. It is simply not worth what it costs at this sample size.

**The group row is a different kind of entry.** `GroupKFold` reports **0.6684**
against a test score of **0.8474** because it asks a different question. Each of
its folds validates on competition levels the model never trained on, while
every level in the test rows also appears in the training rows. So **0.6684** is
the number to quote for a competition level the model has never seen, and
**0.8474** for athletes from the levels this data contains.

So the `absolute error` column is trustworthy exactly where the design and the
test split ask the same question, which is why the first decision in a model
selection workflow is what you are estimating, and only afterwards which
splitter estimates it.

<a class="anchor" id="model-comparison">

## <font color='#E8800A'>4. Comparing Models</font>
</a>

One of the main reasons to use cross-validation is to compare different models fairly on the same data with the same validation strategy. This helps you pick the best algorithm for your specific problem.

In this section, we compare Logistic Regression with the Decision Tree Classifier on one fixed splitter, `RepeatedStratifiedKFold`, which combines the stratification of Section 3.1 with the repetition of Section 3.2.

<a class="anchor" id="decision-tree">

### <font color='#E8800A'>4.1. Decision Tree Classifier</font>
</a>

Let's implement the same cross-validation framework for a Decision Tree and compare its performance with Logistic Regression. `avg_score` takes the model as an argument and builds a fresh `PreparedEstimator` around it per fold, so comparing a second family costs one call rather than a second harness: pass a `DecisionTreeClassifier` where the logistic regression went.

[Back to TOC](#toc)


<a class="anchor" id="comparison-results">

### <font color='#E8800A'>4.2. Model Comparison Results</font>
</a>

Both models are recorded in a new log, `model_log`, which takes only rows measured on `model_cv`, while `frame_log` stays the record of Section 3.

In [ ]:
# One seeded splitter and one log for every model from here on.
model_cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=2, random_state=RANDOM_STATE)
model_log = []

untuned_result = avg_score(model_cv, X_train_val, y_train_val)
record(model_log, untuned_result, X_test, y_test)

__`Step 18`__ Score a `DecisionTreeClassifier(max_depth=5, random_state=RANDOM_STATE)` on `model_cv` with __avg_score__, and record it in `model_log`.

In [ ]:
# DO IT
dt_model = ...  # <-- CODE HERE
dt_result = ...  # <-- CODE HERE
record(...)  # <-- CODE HERE


Both rows were scored on the same ten folds, so the gap between them belongs to
the models. The tree fits its training rows better, **0.8643** against
**0.8573**, and validates worse, **0.8440** against **0.8494**: a wider gap
between training and validation F1, which is how overfitting starts. The test
column reverses the order, **0.8507** against **0.8474**, by less than either
model's fold-to-fold spread, so neither row outranks the other.

<a class="anchor" id="hyperparameter-tuning">

## <font color='#E8800A'>5. Hyperparameter Tuning</font>
</a>

An algorithm is only a set of instructions, and the same instructions given the same data produce very different models depending on how they are configured. Those configuration choices are the **hyperparameters**.

So model selection has two jobs rather than one: find the algorithm that suits the problem, and find the hyperparameters that make it work.

<a class="anchor" id="what-are-hyperparameters">

### <font color='#E8800A'>5.1. What are Hyperparameters?</font>
</a>

Examples:

**Logistic Regression:**
- `C`: Regularization strength (smaller values = stronger regularization)
- `l1_ratio`: Mix of L1 and L2 regularization (0 is L2, 1 is L1)
- `solver`: Optimization algorithm

**Decision Tree:**
- `max_depth`: Maximum depth of the tree
- `min_samples_split`: Minimum samples needed to split a node
- `min_samples_leaf`: Minimum samples required at a leaf
- `criterion`: How to measure split quality ('gini' or 'entropy')

[Back to TOC](#toc)


<a class="anchor" id="grid-search">

### <font color='#E8800A'>5.2. Hyperparameter Search with Holdout Validation</font>
</a>

When tuning hyperparameters, we need to search through different combinations of parameter values to find the best performing model. The two most common approaches to do this are:

1. **Grid Search**: Systematically test all combinations from a predefined grid (some implementations of this method often incorrectly refer to this as **manual search**)
2. **Random Search**: Randomly sample parameter combinations from a predefined grid or distribution of parameters

Let's start with a **grid search using the holdout method** (train-validation-test split).

__`Step 19`__ Define the candidates every search below chooses between: four logistic regressions and four decision trees

In [ ]:
# Define a grid of hyperparameters to search: four logistic regressions and
# four seeded trees, shared by every search below.
candidates = (
    [LogisticRegression(C=C, max_iter=1000) for C in (0.01, 0.1, 1, 10)]
    + [DecisionTreeClassifier(max_depth=depth, min_samples_leaf=leaf,
                              random_state=RANDOM_STATE)
       for depth in (5, None) for leaf in (1, 25)]
)

for candidate in candidates:
    print(repr(candidate))
print(f"\n{len(candidates)} candidates")

__`Step 20`__ Perform grid search using holdout validation

We loop through every candidate of both families:
- We fit preprocessing (scaling, NaN filling) only on **X_train**
- We evaluate each model on **X_val** (which was not used for training or to make preprocessing decisions)
- The **test set is never touched** during hyperparameter selection

In [ ]:
# Manual grid search with holdout validation
holdout_rows = []
for candidate in candidates:
    trained = PreparedEstimator(
        classification_preprocessor(classification_recipe),
        clone(candidate),
    ).fit(X_train, y_train)
    params = candidate.get_params()
    guesses = trained.predict(X_val)
    holdout_rows.append({
        'family': type(candidate).__name__,
        'C': params.get('C'),
        'max_depth': str(params.get('max_depth')),
        'min_samples_leaf': params.get('min_samples_leaf'),
        'train F1': f1_score(y_train, trained.predict(X_train)),
        'validation accuracy': accuracy_score(y_val, guesses),
        'validation precision': precision_score(y_val, guesses),
        'validation recall': recall_score(y_val, guesses),
        'validation F1': f1_score(y_val, guesses),
        'candidate': repr(candidate),
    })
holdout_results = pd.DataFrame(holdout_rows)

# idxmax returns the first maximum, the same tie rule as avg_score.
best = holdout_results['validation F1'].idxmax()
best_candidate = candidates[best]

print("Grid Search Results (Holdout)")
print("=" * 50)
print(holdout_results[['train F1', 'validation accuracy', 'validation precision',
                       'validation recall', 'validation F1', 'candidate']]
      .to_string(index=False, float_format='{:.4f}'.format))
print(f"\nBest candidate: {holdout_results.loc[best, 'candidate']}")
print(f"Best validation F1: {holdout_results.loc[best, 'validation F1']:.4f}")

__`Step 21`__ Visualize the grid search results

In [ ]:
# Plot every candidate of both families in the course colours, training F1 in
# blue and validation F1 in orange, so the gap between the two reads off each
# setting.
fig, (ax_logistic, ax_tree) = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

logistic_rows = holdout_results[holdout_results['family'] == 'LogisticRegression']
ax_logistic.plot(logistic_rows['C'], logistic_rows['train F1'],
                 color=PLOT_BLUE, linewidth=2, marker='o', markersize=8,
                 label='training')
ax_logistic.plot(logistic_rows['C'], logistic_rows['validation F1'],
                 color=PLOT_ORANGE, linewidth=2, marker='o', markersize=8,
                 label='validation')
ax_logistic.set_xscale('log')
ax_logistic.set_xlabel('C (Regularization Parameter)', fontsize=12)
ax_logistic.set_ylabel('F1', fontsize=12)
ax_logistic.set_title('Logistic Regression', fontsize=13, fontweight='bold')
ax_logistic.legend(fontsize=10)

# A depth of None cannot sit on a numeric axis, so each depth is its own line
# style (solid for 5, dashed for None) and the leaf sizes are evenly spaced
# positions.
tree_rows = holdout_results[holdout_results['family'] == 'DecisionTreeClassifier']
leaf_sizes = sorted(tree_rows['min_samples_leaf'].unique())
positions = list(range(len(leaf_sizes)))
for depth, linestyle in (('5', '-'), ('None', '--')):
    rows = tree_rows[tree_rows['max_depth'] == depth]
    ax_tree.plot(positions, rows['train F1'], color=PLOT_BLUE, linewidth=2,
                 marker='o', markersize=8, linestyle=linestyle,
                 label=f'max_depth={depth}, training')
    ax_tree.plot(positions, rows['validation F1'], color=PLOT_ORANGE, linewidth=2,
                 marker='o', markersize=8, linestyle=linestyle,
                 label=f'max_depth={depth}, validation')
ax_tree.set_xticks(positions, [str(int(leaf)) for leaf in leaf_sizes])
ax_tree.set_xlabel('min_samples_leaf', fontsize=12)
ax_tree.set_title('Decision Tree', fontsize=13, fontweight='bold')
ax_tree.legend(fontsize=10)

fig.suptitle('Holdout search: training and validation F1 of every candidate',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In the tree panel the training and validation lines come apart. With no
limit on depth or leaf size the tree scores **1.0000** on its training rows and
**0.8193** on validation: it has memorised the rows it was shown. Requiring 25
rows per leaf closes most of that gap, **0.8751** against **0.8577**, and that
setting wins the holdout search.

The logistic regressions gain training F1 as a larger `C` weakens the
regularization, from **0.8396** at `C=0.01` to **0.8662** at `C=1`, and then
level off at **0.8653**, while their validation F1 peaks at `C=0.1`, **0.8464**,
and falls after it.

__`Step 22`__ Deploy the winner: retrain it on train + validation and evaluate it on the test set

Now that the validation set has selected the best candidate, we can:
1. Retrain on **train + validation** combined (to use more data)
2. Evaluate on the **test set** for an unbiased performance estimate

In [ ]:
# Refit the selected configuration on train+validation rows.
final_model = PreparedEstimator(
    classification_preprocessor(classification_recipe),
    clone(best_candidate),
).fit(X_train_val, y_train_val)

best_score = holdout_results.loc[best, 'validation F1']
test_score = f1_score(y_test, final_model.predict(X_test))

print("Final Model Evaluation")
print(f"Deployed: {holdout_results.loc[best, 'candidate']}")
print(f"Validation F1: {best_score:.4f}")
print(f"Test F1: {test_score:.4f}")
print(f"Difference: {abs(best_score - test_score):.4f}")
print("\nA similar validation and test score is evidence, not a guarantee, of generalization.")

---

## <font color='#E8800A'>5.3. Grid Search with Cross-Validation</font> <a class="anchor" id="grid-search-cv"></a>

Section 5.2 chose its winner on one validation split. Running the same search over folds replaces that single draw with an average, and `avg_score` already does it: given the list of candidates, it returns the one with the highest mean validation F1, refitted on every training row.

__`Step 23`__ Run the same search on `model_cv` with __avg_score__

In [ ]:
# The same eight candidates, scored on model_cv.
search_result = avg_score(model_cv, X_train_val, y_train_val, model=candidates)
print(f"\nWinner: {search_result['name']}")

__`Step 24`__ Deploy the model __avg_score__ returned, and record it in `model_log`

In [ ]:
# avg_score already refitted the winner; record scores it once on the test rows.
record(model_log, search_result, X_test, y_test,
       validation_is="winner's own score")

The same eight candidates, scored on ten folds instead of one split, pick the
same tree. Its validation F1 falls from **0.8577** on one split to **0.8507**
averaged over ten, and the unrestricted tree again scores **1.0000** on its
training rows against **0.7996** on validation.

The search does not separate the two families. The tree wins at **0.8507** and
the best logistic regression, `C=10`, scores **0.8506**, a margin in the fourth
decimal against fold-to-fold spreads of **0.0138** and **0.0172**. The strict
comparison in `avg_score` settles a margin that small, and a different draw of
folds could as easily return a logistic regression.

On the test rows the deployed tree scores **0.8385**, below the **0.8474** of
the untuned logistic regression in the first row of `model_log`. That gap is
smaller than the fold-to-fold spread, on one draw of 800 rows, so it does not
show that tuning hurt; it shows that tuning this grid bought nothing
measurable. The winner's own score, **0.8507**, misses its test F1 by
**0.0122**, the largest miss in the log so far.

---

## <font color='#E8800A'>5.4. Using Pipeline with GridSearchCV</font> <a class="anchor" id="pipeline"></a>

The safest way to use scikit-learn's GridSearchCV (or `RandomizedSearchCV`, which samples the grid instead of trying every cell) is to combine it with **Pipeline**. A Pipeline ensures that all preprocessing steps are applied correctly during cross-validation, preventing data leakage.

__`Step 25`__ Implement Grid Search using Pipeline and GridSearchCV

In [ ]:
# Pipeline is scikit-learn's own pairing of the recipe and the model.
# GridSearchCV refits both steps inside every training fold.
pipeline = Pipeline([
    ('prepare', classification_preprocessor(classification_recipe)),
    ('classifier', LogisticRegression(max_iter=1000)),
])

# The eight candidates of Section 5.2, written as two grids. The 'classifier'
# entry swaps the model itself, and the entries after it vary its settings.
param_grid_pipeline = [
    {'classifier': [LogisticRegression(max_iter=1000)],
     'classifier__C': [0.01, 0.1, 1, 10]},
    {'classifier': [DecisionTreeClassifier(random_state=RANDOM_STATE)],
     'classifier__max_depth': [5, None],
     'classifier__min_samples_leaf': [1, 25]},
]

grid_search = GridSearchCV(
    pipeline,
    param_grid_pipeline,
    cv=model_cv,
    scoring='f1',
    return_train_score=True,
    error_score='raise',
    verbose=1,
    n_jobs=4,
)

print("Running GridSearchCV with Pipeline...")
grid_search.fit(X_train_val, y_train_val)

grid_winner = repr(grid_search.best_estimator_.named_steps['classifier'])
test_score_pipeline = f1_score(y_test, grid_search.predict(X_test))

print("\nGridSearchCV Results:")
print(f"Best candidate: {grid_winner}")
print(f"Best CV F1: {grid_search.best_score_:.4f}")
print(f"Its training F1: "
      f"{grid_search.cv_results_['mean_train_score'][grid_search.best_index_]:.4f}")
print(f"Test F1: {test_score_pipeline:.4f}")

# The same folds, candidates and metric as Section 5.3, so the same answer.
assert grid_winner == search_result['name']
assert np.isclose(grid_search.best_score_, np.mean(search_result['validation']))
assert np.isclose(test_score_pipeline,
                  f1_score(y_test, search_result['model'].predict(X_test)))
print("\nSame winner, cross-validated F1 and test F1 as the Section 5.3 search.")

---

## <font color='#E8800A'>5.5. The Two Execution Schemas</font> <a class="anchor" id="execution-schemas"></a>

| Standard search, as in Sections 5.2 to 5.4 | Cached fold search, `CachedSearchCV` |
|---|---|
| Fold 1, candidate A: fit preprocessing, fit model | Fold 1: fit preprocessing once |
| Fold 1, candidate B: fit the same preprocessing again | Cache training and validation matrices |
| Repeat for every candidate and fold | Run candidates A, B, and the rest on those matrices |

Both schemas keep validation rows out of fitting. The difference is ownership:
the standard interface gives each candidate its own full fitted object, while the
cached one gives the fold one fitted preprocessing object shared by all
candidates with the same preprocessing settings.

[Back to TOC](#toc)

<div class="alert alert-block alert-warning">

**A correct standard search can still repeat work.** When the
complete fitted estimator is cloned for every parameter candidate, the encoder,
fills and scaler are fitted again even when their settings are identical. This is
leakage safe because each fit sees only its training fold, but it is redundant.

Every search so far used that standard implementation. The cached schema keeps
the interface and removes the repetition: each fold fits one copy of every
preprocessing choice, caches its two matrices, then runs every model and
parameter candidate on them. Nested cross-validation runs a whole search inside
every outer fold, so it uses the cached search.

</div>

__`Step 26`__ Run the Section 5.3 search both ways, time them, and
check that only the cost changed.

In [ ]:
# The Section 5.3 search both ways: the candidates fill the Pipeline's
# 'classifier' step and the PreparedEstimator's 'estimator'. Both run on one
# worker, so the times compare the work and not the threads.
standard_template = GridSearchCV(
    pipeline,
    {"classifier": candidates},
    scoring="f1",
    cv=model_cv,
    error_score="raise",
)
cached_template = CachedSearchCV(
    PreparedEstimator(
        classification_preprocessor(classification_recipe),
        LogisticRegression(max_iter=1000),
    ),
    {"estimator": candidates},
    scoring="f1",
    cv=model_cv,
    n_jobs=1,
)


def timed_fit(search):
    """Return a fresh fitted search and its wall-clock duration in seconds."""
    started = perf_counter()
    fitted = clone(search).fit(X_train_val, y_train_val)
    return fitted, perf_counter() - started


standard_search, standard_seconds = timed_fit(standard_template)
cached_search, cached_seconds = timed_fit(cached_template)

same_scores = np.allclose(
    standard_search.cv_results_["mean_test_score"],
    cached_search.cv_results_["mean_test_score"],
)
standard_fits = model_cv.get_n_splits() * len(candidates)
cached_fits = model_cv.get_n_splits()
speed_up = standard_seconds / cached_seconds

assert same_scores, "the efficient execution changed the candidate scores"
assert speed_up > 1.0, "the cached execution must be faster on this benchmark"
# Both schemas crown the Section 5.3 winner, with the same F1.
assert repr(cached_search.best_estimator_.estimator) == search_result["name"]
assert np.isclose(cached_search.best_score_, np.mean(search_result["validation"]))

# Both score columns side by side, one row per candidate.
comparison = pd.DataFrame({
    "standard": standard_search.cv_results_["mean_test_score"],
    "cached": cached_search.cv_results_["mean_test_score"],
})
comparison["difference"] = comparison["standard"] - comparison["cached"]
comparison["candidate"] = [repr(model) for model in candidates]

print(f"standard: {standard_fits} fold preprocessing fits, {standard_seconds:.1f}s")
print(f"cached:   {cached_fits} fold preprocessing fits, {cached_seconds:.1f}s")
print(f"observed speed-up: {speed_up:.2f}x")
print(f"largest difference between the two score columns:"
      f" {comparison['difference'].abs().max():.2e}\n")
print(comparison.to_string(index=False,
                           formatters={"standard": "{:.6f}".format,
                                       "cached": "{:.6f}".format,
                                       "difference": "{:+.2e}".format}))

Every candidate scores the same in both columns to the last digit
printed: the cached schema removes repeated work, not evidence. Had a single row
differed, the speed-up would be worth nothing.

The fit counts explain where the time went: eighty fold preprocessing fits
become ten, while the same eighty fold model fits are still made and scored.
Runtime depends on the machine, but the removed work does not. The cached search
crowns the same tree as Sections 5.3 and 5.4, with the same F1.

---

## <font color='#E8800A'>Gold standard: Nested Cross-Validation</font> <a class="anchor" id="nested-cv"></a>

[Back to TOC](#toc)

Every search above reported the score of its own winner. That score is
optimistic, and the reason is mechanical rather than subtle: the setting was
chosen because it scored highest on those folds, so the number carries whatever
luck those folds happened to hold. A search over eight candidates reports the
largest of eight noisy measurements, and the more candidates it tries, the
further that maximum drifts above the truth.

An estimate free of that luck has to come from rows that took no part in
choosing. **Nested cross-validation** supplies them. An **outer** loop holds a
fold back, the **inner** folds run the entire search on what remains, and the
held-back fold scores the winner it never influenced. Repeat once per outer
fold and average.

Although each outer fold pays for a complete search of its own, which is the
usual objection to the design, expense is not the larger cost. Each fold
searches different training rows, so each may return a different winner, and the
loop therefore ends holding several settings and no mandate to prefer any of
them.

**Nested cross-validation does not select a model. It measures a selection
procedure**, treating the whole prepare-search-refit sequence as the thing under
test, and it answers one question: run this procedure on data like this, and how
good is what comes out?

The procedure measured here is the Section 5.3 search, run as the cached search
of Section 5.5, which returns the same scores and the same winner. The outer
loop uses `model_cv` as well, so the nested row in `model_log` is measured on
the same folds as every other row in it.

__`Step 27`__ Wrap the Section 5.3 search in an outer loop on
`model_cv`, score each outer fold once, and read the estimate.

In [ ]:
# Outer folds from model_cv; inside each, the cached Section 5.3 search.
nested_rows = []

# ===== OUTER LOOP: one estimate of the whole search per fold =====
for outer_fold, (outer_train_idx, outer_test_idx) in enumerate(
    model_cv.split(X_train_val, y_train_val),
    start=1,
):
    X_outer_train = X_train_val.iloc[outer_train_idx]
    X_outer_test = X_train_val.iloc[outer_test_idx]
    y_outer_train = y_train_val.iloc[outer_train_idx]
    y_outer_test = y_train_val.iloc[outer_test_idx]

    # --- inner search: SELECT, using outer-training rows only ------------
    inner = clone(cached_template).fit(X_outer_train, y_outer_train)

    # --- outer step: the refitted winner is scored ONCE ------------------
    nested_rows.append(
        {
            "outer fold": outer_fold,
            "inner F1": inner.best_score_,
            "outer train F1": f1_score(y_outer_train, inner.predict(X_outer_train)),
            "outer F1": f1_score(y_outer_test, inner.predict(X_outer_test)),
            "chosen": repr(inner.best_estimator_.estimator),
        }
    )

nested_results = pd.DataFrame(nested_rows)
print(nested_results.to_string(index=False, float_format="{:.4f}".format))
print(
    "\nNested estimate:",
    f"{nested_results['outer F1'].mean():.4f}",
    "+/-",
    f"{np.std(nested_results['outer F1']):.4f}",
)

**Ten outer folds chose four different candidates, and none of them is the
model you deploy.** Nine chose a logistic regression, at three values of `C`,
and one chose the depth-5 tree, while the search over all of the training rows
returns the tree with 25 rows per leaf. The winner moved across families, not
only across settings.

Four answers are not a tie to break, because the column is a diagnostic rather
than a result. It is a statement about identifiability: a sharply peaked grid
returns the same cell from any training rows, while a flat one returns whichever
cell the noise favoured, and a single search would have printed one confident
answer either way. This grid is flat.

Do not expect the inner score to sit above the outer score fold by fold. The
optimism that nesting removes is a property of the procedure across many
datasets. The model you deploy comes from running the same procedure once more,
on all of the training rows, which is what Section 5.3 already did. That is
legitimate rather than circular, because the estimate was never a claim about a
particular fitted model: it was a claim about the procedure, and the deployed
model is one run of exactly that procedure.

What you may not do is read the `outer F1` column, keep the best fold's setting
and deploy that. Those scores came from rows reserved for grading, so choosing
by them puts the selection back inside the estimate and undoes the design
entirely.

__`Step 28`__ Run the same design in one call, once with each search
from Section 5.5. Both searches are estimators, so `cross_validate` can be the
outer loop. Time the two runs and check that only the cost changed.

In [ ]:
# The same outer folds as the loop above, with each Section 5.5 search as
# the inner loop. Both run on one worker, so the times compare the work and not
# the threads.
def timed_nested(search):
    """Return one nested run's outer scores and its duration in seconds."""
    started = perf_counter()
    scores = cross_validate(search, X_train_val, y_train_val, cv=model_cv,
                            scoring="f1", return_train_score=True)
    return scores, perf_counter() - started


outer_folds = model_cv.get_n_splits()
print(f"standard nested search: {outer_folds * standard_fits} fold preprocessing"
      " fits -- this cell takes about six minutes on my computer.")
standard_outer, standard_nested_seconds = timed_nested(standard_template)
cached_outer, cached_nested_seconds = timed_nested(cached_template)

# Both runs equal the loop above on every outer fold.
for outer in (standard_outer, cached_outer):
    assert np.allclose(outer["test_score"], nested_results["outer F1"])
    assert np.allclose(outer["train_score"], nested_results["outer train F1"])
nested_speed_up = standard_nested_seconds / cached_nested_seconds
assert nested_speed_up > 1.0, "the cached execution must be faster on this benchmark"

print(f"standard: {outer_folds * standard_fits} fold preprocessing fits,"
      f" {standard_nested_seconds:.1f}s")
print(f"cached:   {outer_folds * cached_fits} fold preprocessing fits,"
      f" {cached_nested_seconds:.1f}s")
print(f"observed speed-up: {nested_speed_up:.2f}x\n")
print(pd.DataFrame({
    "outer fold": nested_results["outer fold"],
    "standard outer F1": standard_outer["test_score"],
    "cached outer F1": cached_outer["test_score"],
}).to_string(index=False, float_format="{:.4f}".format))
print(f"\nNested estimate, both ways: {cached_outer['test_score'].mean():.4f}"
      f" +/- {np.std(cached_outer['test_score']):.4f}")

The two runs agree on every outer fold, so here too the cache changes
the cost and nothing else. It saves more here than in Section 5.5, because the
cache turns each search's eighty fold preprocessing fits into ten and the nested
design runs one search per outer fold, so 800 become 100. Any design that
repeats a search multiplies what the cache saves.

__`Step 29`__ Deploy the Section 5.3 model, and record the nested
estimate beside it in `model_log`.

In [ ]:
# The deployed model is the one the Section 5.3 search returned.
nested_result = {
    "frame": type(model_cv).__name__,
    "name": search_result["name"],
    "rows": len(y_train_val),
    "folds": len(nested_results),
    "train": list(nested_results["outer train F1"]),
    "validation": list(nested_results["outer F1"]),
    "model": search_result["model"],
}
print("outer folds chose:", nested_results["chosen"].value_counts().to_dict(), "\n")
record(model_log, nested_result, X_test, y_test, validation_is="nested estimate")

**One procedure, two numbers.** The ten outer models were fitted only to be
scored, and are thrown away: the estimate is their mean outer F1, **0.8513**. So
the last two rows of `model_log` hold the same model and the same test F1,
**0.8385**. Their training and validation F1 differ because the nested row
reports the ten outer winners, **0.8587** and **0.8513**, and not the deployed
tree.

The winner's own score, **0.8507**, is the best of eight measurements on the
folds that chose it, so it is optimistic in expectation; the nested estimate
comes from folds that chose nothing. On this run the nested estimate is the
higher of the two, so one run does not show the optimism: the gap is far inside
the fold-to-fold spread of **0.0161**. The optimism is largest when the
candidates are nearly tied, as they are here, and it grows with the number of
candidates tried. Both rows miss the test F1 for the same reason: the one tree
deployed scored below what the procedure delivers on average, on a single draw
of 800 rows.

The procedure being measured is the recipe, this grid and this search rule
taken together, so widening the grid, moving its range or changing its
resolution changes the thing under test and changes the estimate. **Nested
cross-validation tests whether a parameter grid is worth searching.** That is
the question the inner search cannot ask about itself, because it reports the
best cell it found and has no way to say the cell was not worth finding.

It is also what makes comparison across grids possible. Each arm carries its own
search inside the measurement, so a candidate given two hundred settings cannot
beat one given eight merely by drawing more tickets. Compare two un-nested best
scores and the wider grid usually wins, which is largely a fact about the number
of tickets.

Nested cross-validation puts the whole search at **0.8513**, and the untuned
logistic regression in the first row of `model_log` scores **0.8494** on the
same folds, a gain of two thousandths against a fold-to-fold spread of
**0.0161** in the nested estimate itself. Nine of the ten outer folds chose
a logistic regression, yet the search on every training row deployed a tree,
and no setting won more than four folds. So the grid was worth searching exactly
once, to establish that it was not worth searching again.

**Where does this leave the simpler strategies?** All of them remain useful:
holdout for speed and huge data, K-Fold for robust comparison, the
train/validation/test split as the everyday workflow. Nested CV is the reference
design you reach for when the performance claim itself is what matters!

<div class="alert alert-block alert-success">

**Milestone!** You now command the full assessment toolbox: holdout,
train/validation/test, K-Fold and its stratified/repeated variants,
Leave-One-Out, sklearn's search tools, and nested cross-validation as the
gold standard that keeps selection evidence and estimation evidence apart.

</div>

---

## <font color='#E8800A'>The Complete Model Selection Workflow</font> <a class="anchor" id="workflow"></a>

1. **Choose the validation design** (Sections 2 and 3). Deciding how to split
   the data is not a step before choosing a validation strategy: the splitter is
   the strategy. A single holdout split is the cheapest design and the noisiest.
   K-Fold averages over K of them. Stratified K-Fold holds the class balance
   fixed in every fold. Repeated K-Fold redraws the folds. Group-aware and
   time-aware splitters are required rather than preferred once rows share a
   group or sit in time order, because a random split silently breaks those
   structures. Whichever design you choose, reserve the test rows before
   anything else and hold the design fixed across every candidate you compare.

2. **Compare candidates under that design** (Sections 4 and 5). A model family
   and a hyperparameter setting are the same kind of choice, and validation
   evidence decides both. Read training and validation scores together: a wide
   gap between them is overfitting, and two close but low scores are
   underfitting.

3. **Refit the winner on every row that was available for selection.** The
   search chooses a configuration, and the model you keep is that configuration
   trained once on the combined training and validation rows.

4. **Report the estimate your design actually supports.** One test score is one
   draw and carries the noise of one draw, which is why Section 3.5 would not
   rank designs whose gaps to it were smaller than the fold spread. When the
   performance claim is itself the result, nested cross-validation reports a
   spread and is the estimate to quote.

[Back to TOC](#toc)

# <font color='#E8800A'>Key takeaways</font> <a class="anchor" id="takeaways"></a>
[Back to TOC](#toc)

What this session established:

1. **Selection evidence and estimation evidence are different things.** The
   score that chose a setting is optimistic about that setting in expectation,
   which is the reason nested cross-validation exists at all. On this grid the
   optimism was too small to see, and the nested estimate landed above the
   winner's own score.
2. **A split is a modelling assumption.** Rows that share a group, or that sit
   in time order, break a random split, and the repair is a different splitter,
   not a different model.
3. **Repeating a split is how you learn what one split was worth.** A single
   train/test division is a sample of size one.
4. **A validation design is judged by how close its estimate lands, and one
   test draw is a coarse judge.** Every design in Section 3 printed its
   validation estimate beside a test score, but that score is one draw of 800
   rows: it could not rank the random designs against each other, and it cannot
   judge `GroupKFold`, which asks a different question.
5. **One test set of 800 rows cannot rank models that the folds call a tie.**
   The deployed tree scored below the untuned logistic regression on the test
   rows by less than the fold-to-fold spread, and the nested estimate puts the
   whole search two thousandths above the untuned model.
6. **Put the whole pipeline inside the search.** Anything fitted outside the
   fold is scored on rows it has already seen.
7. **A search should end in a model.** `avg_score` returns the winner refitted
   on every training row, so the configuration it chose is ready to predict new
   rows without a second fit.

---

## <font color='#E8800A'>Optional (Advanced): Scikit-learn Hyperparameter Optimization Tools</font> <a class="anchor" id="optional-tools"></a>

Every search in this notebook tried a fixed list of candidates. scikit-learn provides several other tools for hyperparameter optimization, and they avoid leakage on the same condition `GridSearchCV` met in Section 5.4: the estimator they search holds the preprocessing, so it is refitted inside every fold.

### Available Optimization Methods

| Class | Description | Use Case | Documentation |
|-------|-------------|----------|---------------|
| **GridSearchCV** | Exhaustive search over parameter grid | Small parameter spaces, guaranteed to find best in grid | [Link](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html) |
| **RandomizedSearchCV** | Random sampling from parameter distributions | Large parameter spaces, faster than grid search | [Link](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.RandomizedSearchCV.html) |
| **HalvingGridSearchCV** | Successive halving on parameter grid | Large datasets, eliminates poor candidates early | [Link](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.HalvingGridSearchCV.html) |
| **HalvingRandomSearchCV** | Successive halving with random sampling | Very large parameter spaces | [Link](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.HalvingRandomSearchCV.html) |
| **BayesSearchCV** | Bayesian optimization (requires scikit-optimize) | Expensive model training, smart parameter exploration | [Link](https://scikit-optimize.github.io/stable/modules/generated/skopt.BayesSearchCV.html) |

Beyond scikit-learn, [Optuna](https://optuna.org/) is a dedicated optimisation library whose samplers and pruning strategies go well past a grid; nothing in this course requires it.

[Back to TOC](#toc)